# 2. Data and Clear Data
In the previous notebook, "first analysis data," we examined the data as follows:  
1. Data type  
2. Data issues and problems  
3. Comparing data with each other  
4. Plotting charts from the data  
5. Assessing the data distribution to determine if it is normal or not  
6. Analyzing data behavior before movement, during movement, and after movement  
7. Identifying the type of movement and the dominant foot (which foot bears the most force and pulls the movement toward itself)  
8. Examining movement behavior  

In [47]:
#basic
import os
import math

#analysis
import numpy as np
import pandas as pd

#Statistik
import scipy.stats as stats

#plot
import matplotlib.pyplot as plt
import seaborn as sns

one file for sampel

In [48]:
file_path = r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-11 09-38-11-074_Asphalt.txt"

with open(file_path, 'r', encoding='utf-8') as f:
    first_two_lines = [next(f) for _ in range(2)]

for i, line in enumerate(first_two_lines, 1):
    print(f"line {i}: {line.rstrip()}")

line 1: File: loadsol_25-11-11 09-38-11-074.pdo
line 2: Comment:P02_Asphalt_01


name file

In [49]:
first_two_lines[0][5:].replace(".pdo\n","")

' loadsol_25-11-11 09-38-11-074'

name Type of surface and last number

In [50]:
first_two_lines[1][8:].replace("\n","").split("_")

['P02', 'Asphalt', '01']

In [126]:
file_path =  r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-11 09-38-11-074_Asphalt.txt"


# خواندن 4 سطر اول (نگه می‌داریم برای استخراج نام ستون‌ها)
with open(file_path, "r", encoding="utf-8") as f:
    line1 = f.readline().rstrip("\n")
    line2 = f.readline().rstrip("\n")
    line3 = f.readline().rstrip("\n")
    line4 = f.readline().rstrip("\n")
print(line1)
print(line2)
print(line3)
print(line4)




File: loadsol_25-11-11 09-38-11-074.pdo
Comment:P02_Asphalt_01
	PRY201-L::Heel	PRY201-L::Midfoot	PRY201-L::Forefoot	PRY201-L 		PRY202-R::Forefoot	PRY202-R::Midfoot	PRY202-R::Heel	PRY202-R
Time[secs]	Force[N]	Force[N]	Force[N]	Force[N]	Time[secs]	Force[N]	Force[N]	Force[N]	Force[N]	Time[secs]	AccX	AccY	AccZ	


In [128]:
line3

'\tPRY201-L::Heel\tPRY201-L::Midfoot\tPRY201-L::Forefoot\tPRY201-L \t\tPRY202-R::Forefoot\tPRY202-R::Midfoot\tPRY202-R::Heel\tPRY202-R'

In [160]:
a = np.array(line3.split("\t"))
b = np.array(line4.split("\t"))[0:10]
c = a+b
a_clean = np.char.replace(c, "Force[N]", "")
a_clean

array(['Time[secs]', 'PRY201-L::Heel', 'PRY201-L::Midfoot',
       'PRY201-L::Forefoot', 'PRY201-L ', 'Time[secs]',
       'PRY202-R::Forefoot', 'PRY202-R::Midfoot', 'PRY202-R::Heel',
       'PRY202-R'], dtype='<U18')

In [135]:
col =[]
for i in range(len(line3.split("\t"))):
    col.append(line3.split("\t")[i]+line4.split("\t")[i])

col

# line3.split("\t")
# line4.split("\t")

['Time[secs]',
 'PRY201-L::HeelForce[N]',
 'PRY201-L::MidfootForce[N]',
 'PRY201-L::ForefootForce[N]',
 'PRY201-L Force[N]',
 'Time[secs]',
 'PRY202-R::ForefootForce[N]',
 'PRY202-R::MidfootForce[N]',
 'PRY202-R::HeelForce[N]',
 'PRY202-RForce[N]']

In [ ]:


# مسیر فایل
file_path =  r"C:/Users/user/Desktop/Fatemeh/Analysis_Data/data/raw/Force data (novel loadsol 2)/Force data (novel loadsol 2)/loadsolASCII_25-11-11 09-38-11-074_Asphalt.txt"


# خواندن 4 سطر اول (نگه می‌داریم برای استخراج نام ستون‌ها)
with open(file_path, "r", encoding="utf-8") as f:
    line1 = f.readline().rstrip("\n")
    line2 = f.readline().rstrip("\n")
    line3 = f.readline().rstrip("\n")
    line4 = f.readline().rstrip("\n")

# اطمینان از اینکه طول دو خط عنوان برابر است (برای سادگی پردازش)
maxlen = max(len(line3), len(line4))
line3_p = line3.ljust(maxlen)
line4_p = line4.ljust(maxlen)

# پیدا کردن span های ستون بر اساس وجود کاراکتر غیر-space در یکی از دو خط
spans = []
in_span = False
for i in range(maxlen):
    char_nonspace = (line3_p[i] != " ") or (line4_p[i] != " ")
    if char_nonspace and not in_span:
        start = i
        in_span = True
    elif (not char_nonspace) and in_span:
        end = i  # end exclusive
        spans.append((start, end))
        in_span = False

# اگر تا انتها در span بودیم، آن را اضافه کن
if in_span:
    spans.append((start, maxlen))

# اگر pandas نتواند درست جدا کند، حداقل یک span کامل بساز
if len(spans) == 0:
    spans = [(0, maxlen)]

# خواندن داده‌ها از سطر 5 به بعد بر اساس spans (colspecs)
df = pd.read_fwf(file_path, skiprows=4, colspecs=spans, header=None, encoding="utf-8")

# ساخت نام ستون‌ها از substrings سطر 3 و 4 براساس همان spans
col_names = []
for (s, e) in spans:
    part1 = line3_p[s:e].strip()
    part2 = line4_p[s:e].strip()
    # اگر هر دو خالی بودند (شانسی)، نامی بساز یا شماره ستون بگذار
    if part1 == "" and part2 == "":
        col_names.append(f"col_{s}_{e}")
    else:
        # نحوه ترکیب را اینجا انتخاب کن (مثلاً زیرخط)
        combined = (part1 + "_" + part2).strip("_")
        col_names.append(combined)

df.columns = col_names

print("Detected spans (colspecs):", spans)
print("Column names:")
print(df.columns.tolist())
df.head()



Detected spans (colspecs): [(0, 124)]
Column names:
['PRY201-L::Heel\tPRY201-L::Midfoot\tPRY201-L::Forefoot\tPRY201-L \t\tPRY202-R::Forefoot\tPRY202-R::Midfoot\tPRY202-R::Heel\tPRY202-R_Time[secs]\tForce[N]\tForce[N]\tForce[N]\tForce[N]\tTime[secs]\tForce[N]\tForce[N]\tForce[N]\tForce[N]\tTime[secs]\tAccX\tAccY\tAccZ']


,PRY201-L::Heel\tPRY201-L::Midfoot\tPRY201-L::Forefoot\tPRY201-L \t\tPRY202-R::Forefoot\tPRY202-R::Midfoot\tPRY202-R::Heel\tPRY202-R_Time[secs]\tForce[N]\tForce[N]\tForce[N]\tForce[N]\tTime[secs]\tForce[N]\tForce[N]\tForce[N]\tForce[N]\tTime[secs]\tAccX\tAccY\tAccZ
0,"0,000\t182,040\t162,060\t127,280\t471,380\t0,0..."
1,"0,010\t182,040\t162,060\t127,280\t471,380\t0,0..."
2,"0,020\t185,000\t164,280\t127,280\t476,560\t0,0..."
3,"0,030\t182,040\t162,060\t127,280\t471,380\t0,0..."
4,"0,040\t182,040\t164,280\t127,280\t473,600\t0,0..."


In [66]:
line3

'PRY201-L::Heel\tPRY201-L::Midfoot\tPRY201-L::Forefoot\tPRY201-L \t\tPRY202-R::Forefoot\tPRY202-R::Midfoot\tPRY202-R::Heel\tPRY202-R'